# 03 - Baseline Training

## Objective

Train a baseline YOLO11n object detection model
on the Unified Coastal Waste Dataset (UCWD)
using transfer learning.

This baseline will serve as a reference for future
experiments involving higher input resolutions
and image tiling strategies.

The original dataset splits are retained.
Known spatial and temporal data leakage limitations
must be considered when interpreting model performance.

## 1. Library imports

In [1]:
from pathlib import Path

import torch
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ultralytics import YOLO
from PIL import Image

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42

## 2. Hardware verification

In [2]:
# Check PyTorch and CUDA availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print(f"GPU memory: {gpu_memory:.2f} GB")

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Selected device:", DEVICE)

PyTorch version: 2.15.0+cu132
CUDA available: True
CUDA version: 13.2
GPU: NVIDIA GeForce RTX 3050 4GB Laptop GPU
GPU memory: 4.00 GB
Selected device: 0


## 3. Dataset configuration

In [3]:
# Project directory
PROJECT_DIR = Path(
    r"C:\Users\lenam\Documents\GitHub\plastic-waste-detection"
).resolve()

# Original dataset directory
DATA_DIR = PROJECT_DIR / "data" / "raw" / "ucwd" / "UCWD.yolo"

# Load original configuration
with open(DATA_DIR / "data.yaml", "r", encoding="utf-8") as file:
    original_config = yaml.safe_load(file)

# Create YOLO configuration
dataset_config = {
    "path": DATA_DIR.as_posix(),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(original_config["names"]),
    "names": original_config["names"]
}

# Verify dataset paths
for split in ["train", "val", "test"]:

    split_path = DATA_DIR / dataset_config[split]

    assert split_path.is_dir(), f"Missing directory: {split_path}"

    print(f"{split}: {split_path.exists()}")

# Save configuration file
CONFIG_DIR = PROJECT_DIR / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = CONFIG_DIR / "ucwd.yaml"

with open(DATA_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dataset_config, file, sort_keys=False)

print("\nConfiguration saved:", DATA_YAML)

print("\nDataset configuration:")
print(yaml.safe_dump(dataset_config, sort_keys=False))

train: True
val: True
test: True

Configuration saved: C:\Users\lenam\Documents\GitHub\plastic-waste-detection\configs\ucwd.yaml

Dataset configuration:
path: C:/Users/lenam/Documents/GitHub/plastic-waste-detection/data/raw/ucwd/UCWD.yolo
train: train/images
val: valid/images
test: test/images
nc: 7
names:
- Glass
- Metal
- Packet
- Plastic bag
- Plastic bottle
- Plastic container
- Plastic fragment



## 4. Model initialization

Initialize a YOLO11n object detection model using
pretrained COCO weights to perform transfer learning on the UCWD dataset.

In [4]:
# Load pretrained YOLO11n model
model = YOLO("yolo11n.pt")

# Display model architecture information
model.info()

print("\nInitial number of classes:", len(model.names))

YOLO11n summary: 181 layers, 2,624,080 parameters, 0 gradients, 6.7 GFLOPs

Initial number of classes: 80


## 5. Baseline training

Train YOLO11n using a standard input resolution
of 640 pixels and pretrained weights.

This experiment establishes a reference baseline for subsequent small object detection improvements.

In [5]:
# Define output directory
RESULTS_DIR = PROJECT_DIR / "results" / "training"

# Train baseline model
results = model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=4,
    device=DEVICE,
    optimizer="auto",
    amp=True,
    workers=0,
    seed=RANDOM_SEED,
    patience=10,
    cache=False,
    plots=True,
    project=str(RESULTS_DIR),
    name="yolo11n_baseline_640"
)

Ultralytics 8.4.216  Python-3.11.7 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 3050 4GB Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\lenam\Documents\GitHub\plastic-waste-detection\configs\ucwd.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scal